## Final data manipulation

Picks up where `detrending.ipynb` leaves off. That notebook performs the core quantile-polynomial
de-trending and writes its result, `hourlytemp_detrended_allyears`, to
`outputs/hourlytemp_detrended_allyears.pkl`. This notebook loads that pickle and performs all of the downstream data
manipulation that is *not* part of the de-trending algorithm itself:

- **Section 2** reshapes the de-trended hourly library into per-station CSVs, computes daily
  temperature stats and CDD65/HDD65, rolls those up to annual/monthly totals, and aggregates
  both hourly and daily temperatures up to planning-area (PA) level using station weights.
- **Section 3** derives hourly dew point from de-trended station temperatures and (unadjusted,
  nearest-gridcell) relative humidity, then aggregates dew point up to PA level the same way.

Run `detrending.ipynb` first (through its final "Export de-trended hourly library" cell) before
running this notebook.

# 0. Initial setup

### Import libraries
Import all libraries needed to complete the calculations in this notebook.

In [1]:
import os, sys
os.chdir('/home/jovyan/calvin-sandbox/DFU')
sys.path.insert(0, '/home/jovyan/calvin-sandbox/DFU/updated_nbs')

# from run_pipeline import control_tag
import os

import xarray as xr
import pandas as pd
import numpy as np
import datetime


### Common inputs
Select common inputs and input files needed to process data.

In [2]:
date_stamp = datetime.date.today().strftime('%Y%m%d')
# TAG = control_tag()
RUN = date_stamp

station_list = pd.read_csv("inputs/weather_stations.csv")
station_list = station_list[station_list['STATION'] == 'BAKERSFIELD'].reset_index(drop=True)

station_weights = pd.read_csv("inputs/station_weights_04122019.csv")
station_weights = station_weights[['PA_NAME','STATION','MONTH','PA_WEIGHT']]

variants = pd.read_csv("inputs/variantsID_mapping_20240624.csv", index_col=['scenario','simulation','variantYear'])

simulation_rename = {
    'wrf_ucla_mpi-esm1-2-hr_ssp370_r3i1p1f1': 'WRF_MPI-ESM1-2-HR_r3i1p1f1',
    'wrf_ucla_miroc6_ssp370_r1i1p1f1': 'WRF_MIROC6_r1i1p1f1',
    'wrf_ucla_ec-earth3-veg_ssp370_r1i1p1f1': 'WRF_EC-Earth3-Veg_r1i1p1f1',
    'wrf_ucla_ec-earth3_ssp370_r1i1p1f1': 'WRF_EC-Earth3_r1i1p1f1',
    'wrf_ucla_taiesm1_ssp370_r1i1p1f1': 'WRF_TaiESM1_r1i1p1f1',
}

### Load de-trended hourly library
Read the output of `detrending.ipynb`'s de-trending loop from `outputs/hourlytemp_detrended_allyears.pkl`.

In [3]:
hourlytemp_detrended_allyears = pd.read_pickle(f'outputs/hourlytemp_detrended_allyearsTEMPTAG.pkl')

# 2. Temperature inputs for demand forecasting

### Step 2a. Hourly temperatures by station
For each station, create a csv file with hourly de-trended temperatures across all variants. Each file has ~52 million observations (29 years x 8,760 hours x 204 variants). Data is saved in wide format to reduce file size.

In [4]:
df = hourlytemp_detrended_allyears.stack('STATION').droplevel(level=0, axis=1)
df = df.reorder_levels(['STATION','forecastDateTime_PST','forecastDate_PST','Hour'])
df = df.sort_values(by=['STATION','forecastDateTime_PST'])
df.columns = [f'TEMP_variantID_{col}' for col in df.columns]
#df.to_csv('outputs/hourly/WS_hourly_TEMP_detrended_all_' + RUN + '.csv')
#df.to_hdf('outputs/hourly/WS_hourly_TEMP_detrended_all_' + date_stamp + '.h5', key='data', mode='w')

for station_name, STATION in zip(station_list['station_name'].values, station_list['STATION'].values):
    print(station_name)
    df_station = df[df.index.get_level_values('STATION') == STATION]
    df_station.to_csv('outputs/hourly/temperature/WS_hourly_TEMP_detrended_' + STATION + '_' + RUN + '.csv')

# Delete unused data from memory 
# del df
del df_station

/tmp/ipykernel_745/1409906050.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = hourlytemp_detrended_allyears.stack('STATION').droplevel(level=0, axis=1)


Bakersfield Meadows Field (KBFL)


### Step 2b. Daily temperature stats by station
For each station, create a csv file with daily temperature statistics across all variants.

In [5]:
hourlytemp_daily_input = hourlytemp_detrended_allyears.droplevel(['forecastDate_PST','Hour'], axis = 0)

# Calculate daily temperature stats
dailymaxtemp_detrended = hourlytemp_daily_input.resample('D').max()
dailymintemp_detrended = hourlytemp_daily_input.resample('D').min()
dailymeantemp_detrended = hourlytemp_daily_input.resample('D').mean()

# Drop leap days
dailymaxtemp_detrended = dailymaxtemp_detrended[~((dailymaxtemp_detrended.index.month == 2) & (dailymaxtemp_detrended.index.day == 29))]
dailymintemp_detrended = dailymintemp_detrended[~((dailymintemp_detrended.index.month == 2) & (dailymintemp_detrended.index.day == 29))]
dailymeantemp_detrended = dailymeantemp_detrended[~((dailymeantemp_detrended.index.month == 2) & (dailymeantemp_detrended.index.day == 29))]

# Rename time axis
dailymaxtemp_detrended = dailymaxtemp_detrended.rename_axis(index='forecastDate_PST')
dailymintemp_detrended = dailymintemp_detrended.rename_axis(index='forecastDate_PST')
dailymeantemp_detrended = dailymeantemp_detrended.rename_axis(index='forecastDate_PST')

# Re-format data
dailymaxtemp_detrended = dailymaxtemp_detrended.stack().stack().reorder_levels(['STATION','forecastDate_PST','variantID']).sort_index(level=['STATION','variantID','forecastDate_PST'])
dailymintemp_detrended = dailymintemp_detrended.stack().stack().reorder_levels(['STATION','forecastDate_PST','variantID']).sort_index(level=['STATION','variantID','forecastDate_PST'])
dailymeantemp_detrended = dailymeantemp_detrended.stack().stack().reorder_levels(['STATION','forecastDate_PST','variantID']).sort_index(level=['STATION','variantID','forecastDate_PST'])

# Rename variable names
dailymaxtemp_detrended = dailymaxtemp_detrended.rename(columns={'degF_detrended':'TEMP_MAX'})
dailymintemp_detrended = dailymintemp_detrended.rename(columns={'degF_detrended':'TEMP_MIN'})
dailymeantemp_detrended = dailymeantemp_detrended.rename(columns={'degF_detrended':'TEMP_MEAN'})

# Combine data into a single data frame
WS_daily = dailymaxtemp_detrended.merge(dailymintemp_detrended, left_index=True, right_index=True).merge(dailymeantemp_detrended, left_index=True, right_index=True)

# Calculate simple average of daily high and low temperatures, which is used in CDD/HDD calculations. Note that this is different from the daily mean temperature calculated above.
WS_daily['TEMP_AVG'] = (WS_daily['TEMP_MAX'] + WS_daily['TEMP_MIN']) / 2

# Calculate CDD and HDDs based on 65 degF threshold.
WS_daily['CDD65'] = (WS_daily['TEMP_AVG']-65).clip(lower=0)
WS_daily['HDD65'] = (65-WS_daily['TEMP_AVG']).clip(lower=0)

# Save the file in csv format
WS_daily.to_csv('outputs/daily/WS_daily_TEMP_detrended_all_' + RUN + '.csv')

for station_name, STATION in zip(station_list['station_name'].values, station_list['STATION'].values):
    print(station_name)
    df = WS_daily[WS_daily.index.get_level_values('STATION') == STATION]
    df.to_csv('outputs/daily/WS_daily_TEMP_detrended_' + STATION + '_' + RUN + '.csv')

# Delete unused data from memory 
del dailymaxtemp_detrended
del dailymintemp_detrended
del dailymeantemp_detrended

/tmp/ipykernel_745/2344132255.py:19: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  dailymaxtemp_detrended = dailymaxtemp_detrended.stack().stack().reorder_levels(['STATION','forecastDate_PST','variantID']).sort_index(level=['STATION','variantID','forecastDate_PST'])
/tmp/ipykernel_745/2344132255.py:19: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  dailymaxtemp_detrended = dailymaxtemp_detrended.stack().stack().reorder_levels(['STATION','forecastDate_PST','variantID']).sort_index(level=['STATION','variantID','forecastDate_PST'])
/tmp/ipykernel_745/2344132255.py:20: Futur

Bakersfield Meadows Field (KBFL)


### Step 2c. Annual and monthly CDD/HDDs by station
Calculate CDD65 and HDD65 metrics for all stations and variants. 

In [6]:
TAG = "TEMPTAG"

In [7]:
annual_CDD65 = WS_daily['CDD65'].unstack('variantID').unstack('STATION').resample('Y').sum()
annual_CDD65.index = annual_CDD65.index.year
annual_CDD65 = annual_CDD65.rename_axis(index='forecastYear')
annual_CDD65 = annual_CDD65.stack().reorder_levels(['STATION','forecastYear']).sort_index(level=['STATION', 'forecastYear'])
annual_CDD65.to_csv(f'outputs/annual_CDD65{TAG}.csv')

annual_HDD65 = WS_daily['HDD65'].unstack('variantID').unstack('STATION').resample('Y').sum()
annual_HDD65.index = annual_HDD65.index.year
annual_HDD65 = annual_HDD65.rename_axis(index='forecastYear')
annual_HDD65 = annual_HDD65.stack().reorder_levels(['STATION','forecastYear']).sort_index(level=['STATION', 'forecastYear'])
annual_HDD65.to_csv(f'outputs/annual_HDD65{TAG}.csv')

monthly_CDD65 = WS_daily['CDD65'].unstack('variantID').unstack('STATION').resample('M').sum()
monthly_CDD65 = monthly_CDD65.rename_axis(index='forecastMonth')
monthly_CDD65 = monthly_CDD65.stack().reorder_levels(['STATION','forecastMonth']).sort_index(level=['STATION', 'forecastMonth'])
monthly_CDD65.to_csv(f'outputs/monthly_CDD65{TAG}.csv')

monthly_HDD65 = WS_daily['HDD65'].unstack('variantID').unstack('STATION').resample('M').sum()
monthly_HDD65 = monthly_HDD65.rename_axis(index='forecastMonth')
monthly_HDD65 = monthly_HDD65.stack().reorder_levels(['STATION','forecastMonth']).sort_index(level=['STATION', 'forecastMonth'])
monthly_HDD65.to_csv(f'outputs/monthly_HDD65{TAG}.csv')

/tmp/ipykernel_745/1733074869.py:1: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  annual_CDD65 = WS_daily['CDD65'].unstack('variantID').unstack('STATION').resample('Y').sum()
/tmp/ipykernel_745/1733074869.py:4: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  annual_CDD65 = annual_CDD65.stack().reorder_levels(['STATION','forecastYear']).sort_index(level=['STATION', 'forecastYear'])
/tmp/ipykernel_745/1733074869.py:7: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  annual_HDD65 = WS_daily['HDD65'].unstack('variantID').unstack('STATION').resample('Y').sum()
/tmp/ipykernel_745/1733074869.py:10: FutureWarning: The previous implementation of stack is deprecated and will be remove

### Checkpoint: temporal aggregation
Writes the daily station statistics and the annual/monthly CDD/HDD rollups to `outputs/intermediate/` so the temporal stage can be inspected or reloaded without rerunning Section 2.

In [8]:
import os
import pickle
os.makedirs('outputs/intermediate', exist_ok=True)

temporal_checkpoint = {
    'WS_daily': WS_daily,
    'annual_CDD65': annual_CDD65,
    'annual_HDD65': annual_HDD65,
    'monthly_CDD65': monthly_CDD65,
    'monthly_HDD65': monthly_HDD65,
}
with open(f'outputs/intermediate/temporal_{RUN}.pkl', 'wb') as f:
    pickle.dump(temporal_checkpoint, f)


### Step 2d. Hourly temperatures by planning area
Aggregate hourly temperatures to planning area (PA) level by applying station weights.

In [11]:
WS_hourly_TEMP_all = None

for station_name, STATION in zip(station_list['station_name'].values, station_list['STATION'].values):
    print(station_name)
    WS_hourly_TEMP = pd.read_csv('outputs/hourly/temperature/WS_hourly_TEMP_detrended_' + STATION + '_' + RUN + '.csv')
    WS_hourly_TEMP['forecastDateTime_PST'] = pd.to_datetime(WS_hourly_TEMP['forecastDateTime_PST'])
    WS_hourly_TEMP['forecastDate_PST'] = pd.to_datetime(WS_hourly_TEMP['forecastDate_PST'])
    WS_hourly_TEMP['MONTH'] = WS_hourly_TEMP['forecastDate_PST'].dt.month
    WS_hourly_TEMP = WS_hourly_TEMP.set_index(['STATION','forecastDateTime_PST','forecastDate_PST','MONTH','Hour'])
    WS_hourly_TEMP = WS_hourly_TEMP.astype('float32')
    WS_hourly_TEMP_all = pd.concat([WS_hourly_TEMP_all, WS_hourly_TEMP])

WS_hourly_TEMP_all = WS_hourly_TEMP_all.reset_index()
PA_hourly = WS_hourly_TEMP_all.merge(station_weights, on=['STATION','MONTH'],how='outer')

for i in range(103, 154):
    PA_hourly['TEMP_variantID_' + str(i)] = PA_hourly['TEMP_variantID_' + str(i)] * PA_hourly['PA_WEIGHT']

PA_hourly.drop(['STATION','MONTH','PA_WEIGHT'], axis=1, inplace=True)
PA_hourly = PA_hourly.groupby(['PA_NAME','forecastDateTime_PST','forecastDate_PST','Hour']).sum().reset_index()
PA_hourly = PA_hourly.set_index(['PA_NAME','forecastDateTime_PST','forecastDate_PST','Hour'])
PA_hourly = PA_hourly.astype('float32')
PA_hourly

for pa_name in pd.unique(PA_hourly.index.get_level_values('PA_NAME')):
    df = PA_hourly[PA_hourly.index.get_level_values('PA_NAME') == pa_name]
    df.to_csv('outputs/hourly/temperature/PA_hourly_TEMP_detrended_' + pa_name + '_' + RUN + '.csv')

del WS_hourly_TEMP
del WS_hourly_TEMP_all
del df
#del PA_hourly

Bakersfield Meadows Field (KBFL)


FileNotFoundError: [Errno 2] No such file or directory: 'outputs/hourly/temperature/WS_hourly_TEMP_detrended_BAKERSFIELD_20261015.csv'

### Step 2e. Daily temperature stats by planning area
Two alternative approaches considered: (a) aggregate daily stats, and (b) aggregate hourly, then calculate daily stats.

In [ ]:
# Calculate daily temperature stats (aggregate daily stats)
WS_daily_TEMP_all = None

for station_name, STATION in zip(station_list['station_name'].values, station_list['STATION'].values):
    print(station_name)
    WS_daily_TEMP = pd.read_csv('outputs/daily/WS_daily_TEMP_detrended_' + STATION + '_' + RUN + '.csv')
    WS_daily_TEMP['forecastDate_PST'] = pd.to_datetime(WS_daily_TEMP['forecastDate_PST'])
    WS_daily_TEMP['MONTH'] = WS_daily_TEMP['forecastDate_PST'].dt.month
    WS_daily_TEMP = WS_daily_TEMP.set_index(['STATION','forecastDate_PST','MONTH','variantID'])
    WS_daily_TEMP = WS_daily_TEMP.astype('float32')
    WS_daily_TEMP_all = pd.concat([WS_daily_TEMP_all, WS_daily_TEMP])

WS_daily_TEMP_all = WS_daily_TEMP_all.reset_index()
PA_daily = WS_daily_TEMP_all.merge(station_weights, on=['STATION','MONTH'],how='inner')
PA_daily['TEMP_MAX'] = PA_daily['TEMP_MAX'] * PA_daily['PA_WEIGHT']
PA_daily['TEMP_MIN'] = PA_daily['TEMP_MIN'] * PA_daily['PA_WEIGHT']
PA_daily['TEMP_MEAN'] = PA_daily['TEMP_MEAN'] * PA_daily['PA_WEIGHT']
PA_daily['TEMP_AVG'] = PA_daily['TEMP_AVG'] * PA_daily['PA_WEIGHT']
PA_daily['CDD65'] = PA_daily['CDD65'] * PA_daily['PA_WEIGHT']
PA_daily['HDD65'] = PA_daily['HDD65'] * PA_daily['PA_WEIGHT']

PA_daily.drop(['STATION','MONTH','PA_WEIGHT'], axis=1, inplace=True)
PA_daily = PA_daily.groupby(['PA_NAME','forecastDate_PST','variantID']).sum().reset_index()
PA_daily = PA_daily.sort_values(by=['PA_NAME','variantID','forecastDate_PST'])
PA_daily = PA_daily.set_index(['PA_NAME','forecastDate_PST','variantID'])
PA_daily = PA_daily.astype('float32')

for pa_name in pd.unique(PA_daily.index.get_level_values('PA_NAME')):
    df = PA_daily[PA_daily.index.get_level_values('PA_NAME') == pa_name]
    df.to_csv('outputs/daily/PA_daily_TEMP_detrended_' + pa_name + '_' + RUN + '.csv')

# Delete unused data from memory 
del WS_daily_TEMP
del WS_daily_TEMP_all
del PA_daily
del df

In [ ]:
# Calculate daily temperature stats (aggregate hourly, then calculate daily stats)
PA_daily_TEMPmax = PA_hourly.reset_index().drop(['forecastDateTime_PST','Hour',], axis=1).groupby(['PA_NAME','forecastDate_PST']).max()
PA_daily_TEMPmax = PA_daily_TEMPmax.astype('float32')
PA_daily_TEMPmax = PA_daily_TEMPmax.reset_index().melt(id_vars=['PA_NAME','forecastDate_PST'], var_name='variantID', value_name='TEMP_MAX')
PA_daily_TEMPmax['variantID'] = PA_daily_TEMPmax['variantID'].str.extract(r'(\d+)').astype(int)
PA_daily_TEMPmax = PA_daily_TEMPmax.set_index(['PA_NAME','forecastDate_PST','variantID'])

PA_daily_TEMPmin = PA_hourly.reset_index().drop(['forecastDateTime_PST','Hour',], axis=1).groupby(['PA_NAME','forecastDate_PST']).min()
PA_daily_TEMPmin = PA_daily_TEMPmin.astype('float32')
PA_daily_TEMPmin = PA_daily_TEMPmin.reset_index().melt(id_vars=['PA_NAME','forecastDate_PST'], var_name='variantID', value_name='TEMP_MIN')
PA_daily_TEMPmin['variantID'] = PA_daily_TEMPmin['variantID'].str.extract(r'(\d+)').astype(int)
PA_daily_TEMPmin = PA_daily_TEMPmin.set_index(['PA_NAME','forecastDate_PST','variantID'])

PA_daily_TEMPmean = PA_hourly.reset_index().drop(['forecastDateTime_PST','Hour',], axis=1).groupby(['PA_NAME','forecastDate_PST']).mean()
PA_daily_TEMPmean = PA_daily_TEMPmean.astype('float32')
PA_daily_TEMPmean = PA_daily_TEMPmean.reset_index().melt(id_vars=['PA_NAME','forecastDate_PST'], var_name='variantID', value_name='TEMP_MEAN')
PA_daily_TEMPmean['variantID'] = PA_daily_TEMPmean['variantID'].str.extract(r'(\d+)').astype(int)
PA_daily_TEMPmean = PA_daily_TEMPmean.set_index(['PA_NAME','forecastDate_PST','variantID'])

# Combine data into a single data frame
PA_daily = PA_daily_TEMPmax.merge(PA_daily_TEMPmin, left_index=True, right_index=True).merge(PA_daily_TEMPmean, left_index=True, right_index=True)

# Calculate simple average of daily high and low temperatures, which is used in CDD/HDD calculations. Note that this is different from the daily mean temperature calculated above.
PA_daily['TEMP_AVG'] = (PA_daily['TEMP_MAX'] + PA_daily['TEMP_MIN']) / 2

# Calculate CDD and HDDs based on 65 degF threshold.
PA_daily['CDD65'] = (PA_daily['TEMP_AVG']-65).clip(lower=0)
PA_daily['HDD65'] = (65-PA_daily['TEMP_AVG']).clip(lower=0)

for pa_name in pd.unique(PA_daily.index.get_level_values('PA_NAME')):
    df = PA_daily[PA_daily.index.get_level_values('PA_NAME') == pa_name]
    df.to_csv('outputs/daily/PA_daily_TEMP_detrended_' + pa_name + '_' + RUN + '.csv')

# Delete unused data from memory 
del PA_daily_TEMPmax
del PA_daily_TEMPmin
del PA_daily_TEMPmean
del df

### Checkpoint: spatial aggregation (temperature)
Writes the planning-area hourly and daily temperatures to `outputs/intermediate/`.

In [ ]:
import os
import pickle
os.makedirs('outputs/intermediate', exist_ok=True)

spatial_temperature_checkpoint = {
    'PA_hourly': PA_hourly,
    'PA_daily': PA_daily,
}
with open(f'outputs/intermediate/spatial_temperature_{RUN}.pkl', 'wb') as f:
    pickle.dump(spatial_temperature_checkpoint, f)


# 3. Dew point inputs for demand forecasting

### Step 3a. Hourly dew point by station
For each variant, derive dew point using de-trended station-level temperatures and gridded relative humidity. Applied the same method used by AE team for consistency.  
https://github.com/cal-adapt/climakitae/blob/main/climakitae/tools/derived_variables.py

In [ ]:
for station_name, STATION in zip(station_list['station_name'].values, station_list['STATION'].values):
    print(station_name)

    rh_ds = xr.open_dataset(f'inputs/{station_name}_rh_(new_core).nc')
    rh_ds = rh_ds.rename({'sim': 'simulation'})
    rh_ds = rh_ds.sel(simulation=list(simulation_rename.keys()))
    rh_ds = rh_ds.assign_coords(simulation=[simulation_rename[s] for s in rh_ds['simulation'].values])

    # Re-load hourly de-trended temperatures and reshape the data
    WS_hourly_TEMP = pd.read_csv('outputs/hourly/temperature/WS_hourly_TEMP_detrended_' + STATION + '_' + RUN + '.csv')
    WS_hourly_TEMP['forecastDateTime_PST'] = pd.to_datetime(WS_hourly_TEMP['forecastDateTime_PST'])
    WS_hourly_TEMP['forecastDate_PST'] = pd.to_datetime(WS_hourly_TEMP['forecastDate_PST'])
    WS_hourly_TEMP = WS_hourly_TEMP.set_index(['STATION','forecastDateTime_PST','forecastDate_PST','Hour'])
    WS_hourly_TEMP = WS_hourly_TEMP.astype('float32')
    WS_hourly_TEMP = WS_hourly_TEMP.reset_index().melt(id_vars=['STATION','forecastDateTime_PST','forecastDate_PST','Hour'], var_name='variantID', value_name='temperature')
    WS_hourly_TEMP['variantID'] = WS_hourly_TEMP['variantID'].str.extract(r'(\d+)').astype(int)

    # Add variant info
    WS_hourly_TEMP = WS_hourly_TEMP.merge(variants.reset_index(), on='variantID', how='inner')
    WS_hourly_TEMP['time'] = pd.to_datetime(dict({'year':WS_hourly_TEMP['forecastDateTime_PST'].dt.year + WS_hourly_TEMP['variantYear'],
                                                  'month':WS_hourly_TEMP['forecastDateTime_PST'].dt.month,
                                                  'day':WS_hourly_TEMP['forecastDateTime_PST'].dt.day,
                                                  'hour':WS_hourly_TEMP['forecastDateTime_PST'].dt.hour
                                                 }))

    # Relative humidity: this file was already narrowed to the station's nearest gridcell by
    # the clip processor in "Lumen fetch and localize WRF data (new_core).ipynb", so no
    # manual nearest-gridcell search is needed here (unlike the original notebook's argmin step)
    WS_hourly_humidity = rh_ds['relative_humidity_2m']
    del rh_ds
    WS_hourly_humidity = WS_hourly_humidity.expand_dims(scenario=['Historical + SSP 3-7.0 -- Business as Usual'])
    WS_hourly_humidity = WS_hourly_humidity.to_dataframe(name='relative_humidity').reset_index()
    WS_hourly_humidity = WS_hourly_humidity[['scenario','simulation','time','relative_humidity']]

    # Merge datasets for temperature and relative humidity
    WS_hourly = WS_hourly_TEMP.merge(WS_hourly_humidity, on=['scenario','simulation','time'], how='inner')
    del WS_hourly_TEMP
    del WS_hourly_humidity
    WS_hourly = WS_hourly[['STATION','forecastDateTime_PST','forecastDate_PST','Hour','variantID','temperature','relative_humidity']]
    WS_hourly = WS_hourly.sort_values(by=['STATION','variantID','forecastDateTime_PST'])
    WS_hourly = WS_hourly.set_index(['STATION','forecastDateTime_PST','forecastDate_PST','Hour','variantID'])

    # Calculate dew point based on temperature and relative humidity
    WS_hourly['temperature_K'] = (WS_hourly['temperature'] - 32) * (5/9) + 273.15
    WS_hourly['es'] = 0.611 * np.exp(5423 * ((1 / 273.15) - (1 / WS_hourly['temperature_K'])))
    WS_hourly['e_vap'] = (WS_hourly['es'] * WS_hourly['relative_humidity']) / 100.0
    WS_hourly['dew_point'] = ((1 / 273.15) - 0.0001844 * np.log(WS_hourly['e_vap']/ 0.611)) ** (-1)
    WS_hourly['dew_point'] = (WS_hourly['dew_point'] - 273.15) * (9/5) + 32
    WS_hourly = WS_hourly.astype('float32')

    # Reshape data and save as a CSV file
    WS_hourly = WS_hourly[['dew_point']].unstack('variantID').droplevel(level=0, axis=1)
    WS_hourly.columns = [f'DEWP_variantID_{col}' for col in WS_hourly.columns]
    WS_hourly.to_csv('outputs/hourly/dew_point/WS_hourly_DEWP_derived_' + STATION + '_' + RUN + '.csv')


### Step 3b. Hourly dew point by planning area
Aggregate hourly dew point to planning area (PA) level by applying station weights.

In [ ]:
WS_hourly_DEWP_all = None

for station_name, STATION in zip(station_list['station_name'].values, station_list['STATION'].values):
    print(station_name)
    WS_hourly_DEWP = pd.read_csv('outputs/hourly/dew_point/WS_hourly_DEWP_derived_' + STATION + '_' + RUN + '.csv')
    WS_hourly_DEWP['forecastDateTime_PST'] = pd.to_datetime(WS_hourly_DEWP['forecastDateTime_PST'])
    WS_hourly_DEWP['forecastDate_PST'] = pd.to_datetime(WS_hourly_DEWP['forecastDate_PST'])
    WS_hourly_DEWP['MONTH'] = WS_hourly_DEWP['forecastDate_PST'].dt.month
    WS_hourly_DEWP = WS_hourly_DEWP.set_index(['STATION','forecastDateTime_PST','forecastDate_PST','MONTH','Hour'])
    WS_hourly_DEWP = WS_hourly_DEWP.astype('float32')
    WS_hourly_DEWP_all = pd.concat([WS_hourly_DEWP_all, WS_hourly_DEWP])

WS_hourly_DEWP_all = WS_hourly_DEWP_all.reset_index()
PA_hourly = WS_hourly_DEWP_all.merge(station_weights, on=['STATION','MONTH'],how='outer')

for i in range(103, 154):
    PA_hourly['DEWP_variantID_' + str(i)] = PA_hourly['DEWP_variantID_' + str(i)] * PA_hourly['PA_WEIGHT']

PA_hourly.drop(['STATION','MONTH','PA_WEIGHT'], axis=1, inplace=True)
PA_hourly = PA_hourly.groupby(['PA_NAME','forecastDateTime_PST','forecastDate_PST','Hour']).sum().reset_index()
PA_hourly = PA_hourly.set_index(['PA_NAME','forecastDateTime_PST','forecastDate_PST','Hour'])
PA_hourly = PA_hourly.astype('float32')
PA_hourly

for pa_name in pd.unique(PA_hourly.index.get_level_values('PA_NAME')):
    df = PA_hourly[PA_hourly.index.get_level_values('PA_NAME') == pa_name]
    df.to_csv('outputs/hourly/dew_point/PA_hourly_DEWP_derived_' + pa_name + '_' + RUN + '.csv')

del WS_hourly_DEWP
del WS_hourly_DEWP_all
del df


NameError: name 'station_list' is not defined

### Checkpoint: spatial aggregation (dew point)
Writes the planning-area hourly dew point to `outputs/intermediate/`.

In [ ]:
import os
import pickle
os.makedirs('outputs/intermediate', exist_ok=True)

spatial_dewpoint_checkpoint = {'PA_hourly': PA_hourly}
with open(f'outputs/intermediate/spatial_dewpoint_{RUN}.pkl', 'wb') as f:
    pickle.dump(spatial_dewpoint_checkpoint, f)
